# 🏥 Pediatric RAG Pipeline — מדריך מלא עם מיפוי לדרישות

**מטלת אמצע קורס: Custom RAG Pipeline**  
**קורפוס:** שני ספרי רפואת ילדים באנגלית (Kliegman 2015 + AAP 2022)

---

## מה זה RAG?

**RAG = Retrieval-Augmented Generation** — שיטה שמשלבת שני עולמות:

```
שאלה → [שלב 1: RETRIEVAL] → מצא את קטעי הטקסט הרלוונטיים ביותר מהמסמכים שלנו
                          ↓
                [שלב 2: GENERATION] → שלח את הקטעים + השאלה ל-LLM
                          ↓
                        תשובה מנומקת + ציטוט מקורות
```

### למה RAG ולא סתם ChatGPT?

| בעיה עם LLM רגיל | פתרון RAG |
|---|---|
| לא מכיר את הספרים הספציפיים שלנו | מחפש ישירות בתוך הספרים |
| ממציא עובדות (hallucination) | מחויב לענות רק ממה שנמצא |
| לא יודע לצטט מקורות | מחזיר chunk_id + עמוד |
| לא מעודכן לאחר תאריך ה-cutoff | הקורפוס שלנו תמיד מעודכן |

---

## ✅ מיפוי מלא: כל דרישה במסמך → היכן בנייטבוק

| # | דרישה במסמך | שלב בנייטבוק | פונקציה מרכזית |
|---|---|---|---|
| 1 | Data collection | שלב 0: התקנות + שלב 1: נתיבים | — |
| 2 | Document loading | **שלב 2** | `load_pdf()` |
| 3 | Text cleaning & preprocessing | **שלב 2** | `clean_text()` |
| 4 | Chunking (2 strategies!) | **שלב 3** | `chunk_fixed_size()`, `chunk_paragraph()` |
| 5 | Embedding | **שלב 4** | `embed_chunks()` + SentenceTransformer |
| 6 | Indexing | **שלב 5** | `upsert_collection()` + ChromaDB |
| 7 | Retrieval + metric | **שלב 6** | `retrieve()` |
| 8 | Answer generation + citations | **שלב 7** | `generate_answer()` |
| 9 | Citation of sources | **שלב 8** | `answer()` — source extraction |
| 10 | Evaluation + error analysis | **שלבים 10–12** | `run_evaluation()`, `soft_hit()` |
| — | Gold set (50 שאלות) | שלב 10 | `eval/gold_set.jsonl` |
| — | Ablation study (4 ניסויים) | **שלב 11** | `ablation_retrieve()` |
| — | `answer()` interface | שלב 8 | `answer()` |
| — | MANIFEST.md | `data/MANIFEST.md` | — |

---

## ארכיטקטורת המערכת

```
📚 PDF files  (data/raw/)
    │
    ▼  load_pdf()  → clean_text() → build_page_to_section_map()
📄 Documents   {doc_id, text, metadata}              ← דרישה 2+3
    │
    ▼  chunk_fixed_size()  /  chunk_paragraph()
✂️  Chunks      {chunk_id, doc_id, text, metadata}    ← דרישה 4
    │
    ▼  SentenceTransformer("all-MiniLM-L6-v2").encode()
🔢 Embeddings  [384 floats per chunk, L2-normalized]  ← דרישה 5
    │
    ▼  ChromaDB.upsert()  (hnsw:space=cosine)
🗄️  Vector Index  persisted → index/                  ← דרישה 6

━━━━━━━━━━━━ At query time ━━━━━━━━━━━━

❓ Question
    ▼  .encode() [same model]
🔢 Query vector
    ▼  ChromaDB.query()  → top-k
📋 Chunks  [{chunk_id, text, score, metadata}]         ← דרישה 7
    ▼  generate_answer()  → Claude claude-haiku-4-5
💬 Answer  + [Source: chunk_id]                        ← דרישות 8+9
    ▼  answer()
✅ {answer, sources, retrieved_chunks}                  ← Required Interface
```

---
## שלב 0 — התקנת ספריות

> 📋 **דרישה 1 — Data collection:**  
> הדרישה אומרת שהקורפוס צריך להכיל לפחות 30 עמודים. הקורפוס שלנו: **1,156 עמודים** — עובר בהרבה.  
> "The corpus should contain knowledge where a baseline LLM may be wrong" — ✓ תוכן ספציפי של ספרים שה-LLM לא מכיר

| ספרייה | תפקיד |
|---|---|
| `pymupdf` (fitz) | חילוץ טקסט מ-PDF |
| `sentence-transformers` | מודל ה-embedding |
| `chromadb` | Vector database מקומי |
| `anthropic` | API ל-Claude |
| `tqdm` | progress bars |

In [ ]:
!pip install -q pymupdf chromadb sentence-transformers anthropic tqdm

---
## שלב 1 — ייבוא ספריות והגדרת נתיבים

הנתיבים מחושבים יחסית לתיקיית הפרויקט. הניחו ש-PDFs נמצאים ב-`data/raw/`.

> 💡 **טיפ:** הריצו `python setup_data.py` מתיקיית הפרויקט כדי להעתיק  
> את ה-PDFs אוטומטית מהמיקום הנוכחי שלהם אל `data/raw/`.

In [ ]:
import os, re, json, sys, time, statistics
from pathlib import Path
from collections import Counter

import fitz                                    # PyMuPDF
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import anthropic

PROJECT_ROOT = Path(".").resolve()
DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_PROC    = PROJECT_ROOT / "data" / "processed"
INDEX_DIR    = PROJECT_ROOT / "index"

DATA_PROC.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"PDFs found:   {[p.name for p in DATA_RAW.glob('*.pdf')]}")

---
## שלב 2 — טעינת PDF (Document Loading + Text Cleaning)

> 📋 **דרישות 2+3 — Document Loading + Text Cleaning:**
> 
> המסמך דורש שכל document יכיל:
> ```python
> { "doc_id": str, "text": str, "metadata": dict }
> ```
> ו-metadata יכיל `source`, `page`, `section`, `date` (אם רלוונטי).

### `clean_text(text)` — דרישה 3

| פעולה | regex | למה |
|---|---|---|
| `\n{3,}` → `\n\n` | מקמץ שורות ריקות | מניעת גושי רווח |
| `[ \t]+` → ` ` | רווחים כפולים | ניקוי רווחים |
| `\n[•\s\d]+\n` → `\n` | מספרי עמוד (`• 30 •`) | רעש PDF |
| `\n[•z]\s*\n` → `\n` | תווי בולט ריקים | AAP משתמש ב-`z` כבולט |

### `build_page_to_section_map(doc)` — metadata helper

קורא את ה-TOC (Table of Contents) המובנה ב-PDF ומבצע fill-forward —  
כל עמוד מקבל את שם הפרק האחרון שנפתח לפניו.

**דוגמה:**
```
TOC: p242 → "Case 16: Simon, Febrile Seizure"
     p268 → "Case 18: Payton, Throat Pain"

→ עמודים 242–267 כולם יקבלו section = "Case 16: Simon..."
```

### `load_pdf(pdf_path, doc_id_prefix)` — דרישה 2

**קלט:**  
- `pdf_path` — נתיב ל-PDF  
- `doc_id_prefix` — קידומת קצרה (`"aap"` / `"kliegman"`)

**תהליך:**
```
for pg_idx in range(len(doc)):
    text = doc[pg_idx].get_text()   # חילוץ גולמי
    text = clean_text(text)          # ניקוי (דרישה 3)
    if len(text) < 50: skip          # דלג תמונות/שערים ריקים
    doc_id = f"{prefix}_p{pg_idx+1:04d}"  # e.g. "aap_p0051"
    append({doc_id, text, metadata})
```

**פלט:** רשימה של מסמכים, כל אחד עומד בדיוק בפורמט שהמטלה דורשת.

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 3: Text Cleaning
# ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """
    מנקה טקסט גולמי שחולץ מ-PDF.
    מסיר: שורות ריקות מרובות, רווחים כפולים,
           מספרי עמוד ('• 30 •'), בולטים ריקים.
    """
    text = re.sub(r'\n{3,}', '\n\n', text)     # max 2 newlines in a row
    text = re.sub(r'[ \t]+', ' ', text)          # collapse horizontal whitespace
    text = re.sub(r'\n[•\s\d]+\n', '\n', text)  # page-number noise lines
    text = re.sub(r'\n[•z]\s*\n', '\n', text)   # empty bullets (z = AAP bullet char)
    return text.strip()


# ─────────────────────────────────────────────────────────────
# עזר: מיפוי עמוד → שם פרק
# ─────────────────────────────────────────────────────────────
def build_page_to_section_map(doc) -> dict:
    """
    בונה dict {page_index: chapter_title} מה-TOC המובנה של ה-PDF.
    מבצע fill-forward כדי שכל עמוד ישא את שם הפרק האחרון.

    קלט:  fitz.Document
    פלט:  {int: str}  (0-based page index → section title)
    """
    toc = doc.get_toc()   # [[level, title, page_1based], ...]
    raw_map = {page - 1: title.replace('\n', ' ').strip()
               for _, title, page in toc if page > 0}
    filled, current = {}, 'Front Matter'
    for pg in range(len(doc)):
        if pg in raw_map:
            current = raw_map[pg]
        filled[pg] = current
    return filled


# ─────────────────────────────────────────────────────────────
# דרישה 2: Document Loading
# ─────────────────────────────────────────────────────────────
def load_pdf(pdf_path, doc_id_prefix: str) -> list:
    """
    טוען PDF ומחזיר רשימת מסמכים ברמת עמוד.

    קלט:
        pdf_path       — Path או str ל-PDF
        doc_id_prefix  — קידומת קצרה ('aap' / 'kliegman')

    פלט: list of
        {
            'doc_id':   'aap_p0051',
            'text':     '...',          ← cleaned text (דרישה 3)
            'metadata': {
                'source':        'AAP_Case-Based.pdf',
                'page':          51,
                'section':       'Case 3: Baby Girl Smith',
                'doc_id_prefix': 'aap'
            }
        }

    עמודים עם פחות מ-50 תווים (תמונות, שערים) — מדולגים.
    """
    path = Path(pdf_path)
    doc  = fitz.open(str(path))
    section_map = build_page_to_section_map(doc)
    docs = []
    for pg_idx in range(len(doc)):
        text = clean_text(doc[pg_idx].get_text())
        if len(text) < 50:
            continue
        docs.append({
            'doc_id': f'{doc_id_prefix}_p{pg_idx + 1:04d}',
            'text':   text,
            'metadata': {
                'source':        path.name,
                'page':          pg_idx + 1,
                'section':       section_map.get(pg_idx, 'Unknown'),
                'doc_id_prefix': doc_id_prefix,
            }
        })
    doc.close()
    return docs


# ─────────────────────────────────────────────────────────────
# עזר: שמירה ל-JSONL
# ─────────────────────────────────────────────────────────────
def save_jsonl(records, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'  שמור: {path}  ({len(records)} רשומות)')

print('פונקציות טעינה הוגדרו ✓')

In [ ]:
# טעינת שני הספרים
kliegman_docs = load_pdf(
    DATA_RAW / 'Kliegman_Pediatric Decision-Making Strategies_2015.pdf',
    'kliegman'
)
aap_docs = load_pdf(
    DATA_RAW / 'A Case-Based Educational Guide-American Academy of Pediatrics (2022).pdf',
    'aap'
)
all_docs = kliegman_docs + aap_docs

print(f'Kliegman: {len(kliegman_docs)} עמודים')
print(f'AAP:      {len(aap_docs)} עמודים')
print(f'סה"כ:    {len(all_docs)} עמודים  (דרישה: ≥30)')

# הצגת מסמך לדוגמה — עמוד 51 (Baby Girl Smith)
sample = aap_docs[50]
print(f'\n--- דוגמה: {sample["doc_id"]} ---')
print(f'section: {sample["metadata"]["section"]}')
print(sample['text'][:400])

---
## שלב 3 — Chunking: פיצול מסמכים לקטעים

> 📋 **דרישה 4 — Chunking:**  
> "You must experiment with **at least two chunking strategies**"  
> "In your report, explain: chunk size, overlap size, why you chose this strategy, one example where chunking helped or hurt retrieval"
> 
> פורמט הפלט הנדרש לכל chunk:
> ```python
> { "chunk_id": str, "doc_id": str, "text": str, "metadata": dict }
> ```

### אסטרטגיה 1: `chunk_fixed_size()` — Fixed-Size Character Chunking

**פרמטרים:**
- `chunk_size=400` — מקסימום 400 תווים לכל קטע
- `overlap=50` — 50 תווים חוזרים בין קטעים סמוכים (מניע חיתוך של משפטים)

**אלגוריתם:**
```python
step = chunk_size - overlap  # = 350
start = 0
while start < len(text):
    chunk = text[start : start+400]
    chunk_id = f"{doc_id}_fixed_{idx:03d}"
    start += 350   # ← overlap: 50 תווים נכפלים
```

**מתי עדיף:** ספר Kliegman — עמודים קצרים עם רשימות אבחנות מבדלות.  
**דוגמה מהדרישה:** chunk 300 vs 700 — ניסוי האבלציה בשלב 11.

### אסטרטגיה 2: `chunk_paragraph()` — Paragraph-Aware Chunking

**פרמטרים:**
- `min_len=100` — פסקה קצרה מ-100 תווים → ממוזגת עם הבאה
- `max_len=700` — פסקה ארוכה מ-700 תווים → מפוצלת לפי סיומי משפטים

**אלגוריתם:** split על `\n\n` (פסקאות) ← merge קצרות ← split ארוכות לפי `.?!`

**מתי עדיף:** ספר AAP — כל Q&A הוא פסקה שלמה. שמירה על גבולות פסקה שומרת על ההיגיון הקליני.

**השפעה על retrieval (דוגמה מהמטלה):**

| שאלה | Fixed-size | Paragraph |
|---|---|---|
| "Discharge criteria febrile seizure?" | ✓ מחזיר קטע בדיוק מהעמוד | ✓ מחזיר Q&A שלם |
| "Describe CBC results" (span several paragraphs) | ✗ חוצה באמצע תוצאה | ✓ שומר פסקת תוצאות שלמה |

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 4: Chunking — אסטרטגיה 1
# ─────────────────────────────────────────────────────────────
def chunk_fixed_size(documents: list,
                     chunk_size: int = 400,
                     overlap: int = 50) -> list:
    """
    אסטרטגיה 1 — Fixed-Size Character Chunking.

    קלט:
        documents  — list of {doc_id, text, metadata}  (פלט load_pdf)
        chunk_size — תווים מקסימלים לקטע (ברירת מחדל: 400)
        overlap    — תווים חוזרים בין קטעים סמוכים (ברירת מחדל: 50)

    פלט: list of
        {
            'chunk_id':  'aap_p0051_fixed_002',   ← doc_id + strategy + index
            'doc_id':    'aap_p0051',
            'text':      '...',                    ← max 400 תווים
            'metadata':  {..., 'chunk_strategy': 'fixed', 'chunk_index': 2}
        }

    הגיון ה-overlap:
        step = chunk_size - overlap = 350
        chunk_0: text[0:400]
        chunk_1: text[350:750]   ← 50 תווים חוזרים = context bridge
        chunk_2: text[700:1100]
    """
    chunks, step = [], chunk_size - overlap
    for doc in documents:
        text, start, idx = doc['text'], 0, 0
        while start < len(text):
            ct = text[start:start + chunk_size].strip()
            if len(ct) > 30:
                chunks.append({
                    'chunk_id': f"{doc['doc_id']}_fixed_{idx:03d}",
                    'doc_id':   doc['doc_id'],
                    'text':     ct,
                    'metadata': {**doc['metadata'],
                                 'chunk_strategy': 'fixed',
                                 'chunk_index': idx},
                })
            start += step
            idx   += 1
    return chunks


# ─────────────────────────────────────────────────────────────
# דרישה 4: Chunking — אסטרטגיה 2
# ─────────────────────────────────────────────────────────────
def chunk_paragraph(documents: list,
                    min_len: int = 100,
                    max_len: int = 700) -> list:
    """
    אסטרטגיה 2 — Paragraph-Aware Chunking.

    קלט:
        documents  — list of {doc_id, text, metadata}
        min_len    — מינימום תווים לקטע (קצרים נמזגים עם הבא)
        max_len    — מקסימום תווים לקטע (ארוכים נפוצלים לפי .?!)

    אלגוריתם:
        1. split(\n\n) → רשימת פסקאות
        2. ממזג פסקאות עד שמגיעים ל-min_len
        3. אם מגיעים ל-max_len — מפצל לפי סיום משפט

    יתרון על fixed: שומר על גבולות לוגיים של Q&A בספר AAP.
    """
    def split_long(text, max_len):
        sents = re.split(r'(?<=[.!?])\s+', text)
        parts, buf = [], ''
        for s in sents:
            if len(buf) + len(s) + 1 <= max_len:
                buf = (buf + ' ' + s).strip()
            else:
                if buf: parts.append(buf)
                buf = s
        if buf: parts.append(buf)
        return parts or [text[:max_len]]

    chunks = []
    for doc in documents:
        paras  = [p.strip() for p in re.split(r'\n{2,}', doc['text']) if p.strip()]
        merged, buf = [], ''
        for para in paras:
            candidate = (buf + '\n\n' + para).strip() if buf else para
            if len(candidate) <= max_len:
                buf = candidate
                if len(buf) >= min_len:
                    merged.append(buf)
                    buf = ''
            else:
                if buf: merged.append(buf); buf = ''
                merged.extend(split_long(para, max_len))
        if buf: merged.append(buf)
        for idx, ct in enumerate(merged):
            if len(ct) > 30:
                chunks.append({
                    'chunk_id': f"{doc['doc_id']}_para_{idx:03d}",
                    'doc_id':   doc['doc_id'],
                    'text':     ct,
                    'metadata': {**doc['metadata'],
                                 'chunk_strategy': 'paragraph',
                                 'chunk_index': idx},
                })
    return chunks

print('פונקציות chunking הוגדרו ✓')

In [ ]:
print('מפצל קטעים...')
fixed_chunks = chunk_fixed_size(all_docs, chunk_size=400, overlap=50)
para_chunks  = chunk_paragraph(all_docs)

print(f'Fixed-size chunks: {len(fixed_chunks):,}')
print(f'Paragraph chunks:  {len(para_chunks):,}')

# הצגת קטע לדוגמה
ex = fixed_chunks[1000]
print(f'\n--- דוגמה: fixed chunk ---')
print(f'chunk_id:     {ex["chunk_id"]}')
print(f'doc_id:       {ex["doc_id"]}')
print(f'עמוד:         {ex["metadata"]["page"]} | section: {ex["metadata"]["section"][:55]}')
print(f'אורך בתווים:  {len(ex["text"])}')
print(ex['text'][:300])

# שמירה לדיסק (data/processed/) — reproducibility requirement
save_jsonl(fixed_chunks, DATA_PROC / 'chunks_fixed.jsonl')
save_jsonl(para_chunks,  DATA_PROC / 'chunks_paragraph.jsonl')

---
## שלב 4 — Embedding: הפיכת טקסט לוקטורים

> 📋 **דרישה 5 — Embedding:**  
> "Create embeddings for your chunks and store them in a vector index."  
> נלמד: מהו embedding, למה L2 normalization, ומה ההבדל בין מודלים.

### מה זה Embedding?

מיפוי טקסט → וקטור של מספרים כך שטקסטים **דומים במשמעות** קרובים במרחב:

```
'febrile seizure discharge'     → [0.12, -0.34, 0.89, ...]  384 מספרים
'criteria release child fever'  → [0.11, -0.36, 0.91, ...]  ← קרוב!
'pasta carbonara recipe'        → [-0.78, 0.22, -0.11, ...] ← רחוק
```

### `embed_chunks(model, chunks, batch_size=128)`

**קלט:**
- `model` — SentenceTransformer object
- `chunks` — list of chunk dicts (מפלט chunking)
- `batch_size` — כמה קטעים לעבד ביחד (מאזן מהירות/זיכרון)

**פעולה:**
1. מחלץ רשימת `text` מה-chunks
2. מפעיל `model.encode()` עם `normalize_embeddings=True`
3. מחזיר `list[list[float]]` — כל פריט הוא וקטור של 384 floats

**`normalize_embeddings=True` — למה?**  
L2-normalization מחלק כל וקטור באורכו → כל וקטור בעל אורך 1.  
תוצאה: `dot_product(A,B) == cosine_similarity(A,B)` → מהיר יותר ב-ChromaDB.

### המודל: `all-MiniLM-L6-v2`

| מאפיין | ערך |
|---|---|
| פרמטרים | 22M (קטן, מהיר) |
| פלט | 384 מימדים |
| שפה | אנגלית (מתאים לקורפוס שלנו) |
| ביצועים | MTEB Retrieval: 49.3 — ביצועים טובים לגודל |
| זמן | ~5-10 דקות לכל ~15,000 קטעים על CPU |

**⏱ זמן מוערך: 5-10 דקות לכל הקורפוס**

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 5: Embedding
# ─────────────────────────────────────────────────────────────
print('טוען מודל embedding: all-MiniLM-L6-v2 ...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('המודל נטען ✓')


def embed_chunks(model, chunks: list, batch_size: int = 128) -> list:
    """
    מחשב embeddings לרשימת קטעים.

    קלט:
        model      — SentenceTransformer object
        chunks     — list of {chunk_id, text, ...}  (פלט chunking)
        batch_size — גודל batch (128 = ברירת מחדל, מאזן מהירות/RAM)

    פלט: list[list[float]]  —  אחד לכל chunk, אורך 384
         הוקטורים L2-נורמליזדים: אורך=1, dot_product = cosine_similarity

    סיבת normalize_embeddings=True:
        ChromaDB מחשב cosine distance עם hnsw:space=cosine.
        נורמליזציה מבטיחה שה-similarity scores יהיו ב-[0,1].
    """
    texts = [c['text'] for c in chunks]
    print(f'  מחשב embeddings ל-{len(texts):,} קטעים (batch={batch_size})...')
    t0 = time.time()
    vecs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f'  ✓ shape: {vecs.shape}  |  זמן: {time.time()-t0:.1f}s')
    return vecs.tolist()


print('\n=== Fixed-size chunks ===')
fixed_embeddings = embed_chunks(model, fixed_chunks)

print('\n=== Paragraph chunks ===')
para_embeddings  = embed_chunks(model, para_chunks)

---
## שלב 5 — Indexing: אחסון ב-ChromaDB

> 📋 **דרישה 6 — Indexing:**  
> "Your `build_index.py` script should build the index from scratch."  
> "The script should be reproducible — if we delete the index and run again, it should rebuild the same index."

### `upsert_collection(client, name, chunks, embeddings)`

**קלט:**
- `client` — ChromaDB PersistentClient
- `name` — שם ה-collection ("pediatric_fixed" / "pediatric_paragraph")
- `chunks` — list of chunk dicts
- `embeddings` — list of vectors (מ-`embed_chunks`)

**תהליך:**
```
1. delete_collection(name) — מחיקה אם קיים (לריצה חוזרת = reproducibility)
2. create_collection(name, {hnsw:space: cosine})  ← מרחב חיפוש
3. flatten metadata — ChromaDB מקבל str/int/float/bool בלבד
4. upsert() בbatches של 2000 (מגבלת ChromaDB)
```

**Reproducibility:** הפעלה חוזרת תמחק ותבנה מחדש את ה-collections —  
אותו קוד + אותם PDFs → אותו index בדיוק.

### HNSW — Hierarchical Navigable Small World

האלגוריתם מאחורי ChromaDB:
- בונה **גרף מרובד** — כל וקטור מחובר לשכניו הקרובים
- חיפוש מתחיל בשכבה עליונה (sparse) ומצמצם לאזור רלוונטי
- **O(log n)** לחיפוש לעומת O(n) בחיפוש כוח גס
- `hnsw:space=cosine` → מדד המרחק הוא cosine distance

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 6: Indexing
# ─────────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(
    path=str(INDEX_DIR),                        # ← persistent: שמור על הדיסק
    settings=Settings(anonymized_telemetry=False),
)


def upsert_collection(client, name: str, chunks: list, embeddings: list):
    """
    יוצר (או מאפס) ChromaDB collection ומוסיף את כל הקטעים.

    קלט:
        client     — chromadb.PersistentClient
        name       — שם ה-collection
        chunks     — list of chunk dicts  (פלט chunking)
        embeddings — list of float vectors  (פלט embed_chunks)

    פלט: chromadb.Collection object

    Reproducibility: מוחק ויוצר מחדש — אותה קריאה תמיד → אותה תוצאה.
    ChromaDB metadata: כל ערך חייב להיות str/int/float/bool.
                       ערכים מסוג אחר ממופים ל-str.
    """
    try: client.delete_collection(name)    # ← reproducibility
    except: pass

    col = client.create_collection(
        name=name,
        metadata={'hnsw:space': 'cosine'},  # ← cosine similarity
    )

    def flatten(m):
        return {k: (str(v) if not isinstance(v, (str, int, float, bool)) else v)
                for k, v in m.items()}

    BATCH = 2000                            # ChromaDB upsert limit
    for i in range(0, len(chunks), BATCH):
        sl = slice(i, i + BATCH)
        col.upsert(
            ids        = [c['chunk_id'] for c in chunks[sl]],
            embeddings = embeddings[sl],
            documents  = [c['text'] for c in chunks[sl]],
            metadatas  = [flatten({**c['metadata'], 'doc_id': c['doc_id']})
                          for c in chunks[sl]],
        )
        print(f'  upserted {min(i+BATCH,len(chunks))}/{len(chunks)}', end='\r')
    print(f"\n  '{name}': {col.count()} chunks ✓")
    return col


print('בונה index...')
col_fixed = upsert_collection(chroma_client, 'pediatric_fixed',     fixed_chunks, fixed_embeddings)
col_para  = upsert_collection(chroma_client, 'pediatric_paragraph', para_chunks,  para_embeddings)
print('\nIndex נבנה בהצלחה ✓  (שמור ב-index/)')

---
## שלב 6 — Retrieval: חיפוש קטעים רלוונטיים

> 📋 **דרישה 7 — Retrieval:**  
> "Implement a retriever that receives a question and returns the top-k most relevant chunks."  
> Minimum requirement: `retrieve(query: str, k: int = 5) -> list[dict]`  
> Each result must include: `{chunk_id, text, score, metadata}`  
> "You should report at least one retrieval metric."

### `retrieve(query, k=5, strategy='fixed')`

**קלט:**
- `query` — שאלה בשפה טבעית
- `k` — מספר קטעים להחזיר (ברירת מחדל: 5)
- `strategy` — `'fixed'` או `'paragraph'`

**תהליך מפורט:**
```
1. model.encode([query], normalize_embeddings=True)
   → query_vector [384 floats]

2. col.query(query_embeddings=..., n_results=k)
   → ChromaDB מחזיר ids, documents, metadatas, distances

3. המרת distance → similarity:
   ChromaDB מחזיר cosine DISTANCE ∈ [0, 2]
   cosine SIMILARITY = 1 - distance / 2  ∈ [0, 1]
   (עבור L2-normalized vectors)

4. מרכיב dict לכל תוצאה: {chunk_id, text, score, metadata}
```

**פלט** — בדיוק כנדרש במסמך:
```python
[
    {
        'chunk_id': 'aap_p0251_fixed_000',
        'text':     'The patient has returned to baseline...',
        'score':    0.8740,     ← cosine similarity
        'metadata': {'source': 'AAP...pdf', 'page': 251, ...}
    },
    ...
]
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 7: Retrieval
# ─────────────────────────────────────────────────────────────
def retrieve(query: str, k: int = 5, strategy: str = 'fixed') -> list:
    """
    מחזיר top-k קטעים הרלוונטיים ביותר לשאלה.

    קלט:
        query    — שאלה בשפה טבעית
        k        — מספר קטעים להחזיר (ברירת מחדל: 5)
        strategy — 'fixed' → col_fixed | 'paragraph' → col_para

    תהליך:
        1. embed השאלה באותו מודל (normalize_embeddings=True)
        2. ChromaDB cosine search → top-k
        3. המרה: cosine_distance [0,2] → cosine_similarity [0,1]
           score = 1.0 - distance / 2.0

    פלט: list[dict] כנדרש במסמך:
        {
            'chunk_id': str,
            'text':     str,
            'score':    float,   ← cosine similarity
            'metadata': dict
        }
    """
    col   = col_fixed if strategy == 'fixed' else col_para
    q_vec = model.encode([query], normalize_embeddings=True).tolist()

    res = col.query(
        query_embeddings=q_vec,
        n_results=k,
        include=['documents', 'metadatas', 'distances'],
    )

    return [
        {
            'chunk_id': cid,
            'text':     text,
            'score':    round(1.0 - dist / 2.0, 4),
            'metadata': meta,
        }
        for cid, text, meta, dist in zip(
            res['ids'][0], res['documents'][0],
            res['metadatas'][0], res['distances'][0]
        )
    ]

print('retrieve() הוגדרה ✓')

In [ ]:
# הדגמה: retrieval לשאלה על febrile seizure
query = 'What are the discharge criteria for a child with febrile seizure?'
results = retrieve(query, k=5, strategy='fixed')

print(f'Query: {query}\n')
print(f'{"chunk_id":<38} {"score":>6}  p    section')
print('-' * 80)
for r in results:
    print(f'{r["chunk_id"]:<38} {r["score"]:>6.3f}  '
          f'{r["metadata"]["page"]:>3}  {r["metadata"]["section"][:38]}')

print(f'\n--- תוכן הקטע הראשון (score={results[0]["score"]}) ---')
print(results[0]['text'][:500])

---
## שלב 7 — Answer Generation עם Claude

> 📋 **דרישה 8 — Answer Generation:**  
> "Use an LLM to generate an answer from the retrieved chunks."  
> כללים נדרשים:
> 1. Answer only from the retrieved context
> 2. If the answer is not in the context, say so
> 3. Cite the source chunks used in the answer
> 4. Avoid unsupported claims

### `generate_answer(question, retrieved_chunks, model_name)`

**קלט:**
- `question` — השאלה המקורית
- `retrieved_chunks` — פלט `retrieve()` — רשימת קטעים עם scores
- `model_name` — `"claude-haiku-4-5-20251001"` (מהיר + זול)

**בניית ה-context block:**
```
[aap_p0251_fixed_000] (source: AAP_Case-Based.pdf, page 251)
The patient has returned to baseline neurologic status...

---

[aap_p0250_fixed_002] (source: AAP_Case-Based.pdf, page 250)
Febrile seizures typically occur in children aged 6 months...
```

**System Prompt — למה כל כלל?**

| כלל | מטרה | אם לא היה |
|---|---|---|
| Answer only from context | מניעת hallucination | LLM ישלים ממידע כללי |
| Say "not found" if missing | Honest negative | LLM יסתמן עצמו |
| Cite [Source: chunk_id] | Traceability | לא ניתן לאמת |
| No external knowledge | Grounding | ערבוב מקורות |

In [ ]:
# הגדרת ה-API key
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # ← הדבק כאן

if not os.environ.get('ANTHROPIC_API_KEY'):
    print('⚠️  ANTHROPIC_API_KEY לא מוגדר!')
    print('   הגדר: os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."')
else:
    print('✓ ANTHROPIC_API_KEY נמצא')

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישות 8+9: Answer Generation + Citation
# ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a pediatric medicine question-answering assistant.
Answer questions using ONLY the provided context passages.
Rules (required by assignment):
1. Base your answer STRICTLY on the retrieved context. [דרישה 8.1]
2. If the answer is not found: say 'The information was not found in the provided sources.' [דרישה 8.2]
3. After your answer, cite chunk IDs: [Source: chunk_id_1, chunk_id_2] [דרישה 8.3 + 9]
4. Do NOT add knowledge from outside the context. [דרישה 8.4]
"""


def generate_answer(question: str,
                    retrieved_chunks: list,
                    model_name: str = 'claude-haiku-4-5-20251001') -> str:
    """
    שולח שאלה + קטעים ל-Claude ומחזיר תשובה מנומקת עם ציטוטים.

    קלט:
        question         — שאלת המשתמש
        retrieved_chunks — list of {chunk_id, text, score, metadata}
                           (פלט retrieve())
        model_name       — מודל Anthropic (ברירת מחדל: claude-haiku-4-5)

    פלט: str — תשובה הכוללת:
        - תשובה מנומקת המבוססת אך ורק על ה-context
        - [Source: chunk_id_1, chunk_id_2] בסוף

    מבנה ה-prompt:
        System: כללי grounding (ראה SYSTEM_PROMPT)
        User:   Question: ...\n\nContext:\n[chunk_id] (page X)\ntext...\n\nAnswer:
    """
    client = anthropic.Anthropic()

    # בניית context block — כל קטע עם ID + מקור
    ctx_parts = [
        f"[{c['chunk_id']}] (source: {c['metadata'].get('source','')}, "
        f"page {c['metadata'].get('page','')})\n{c['text']}"
        for c in retrieved_chunks
    ]
    context_str = '\n\n---\n\n'.join(ctx_parts)

    user_msg = (
        f'Question:\n{question}\n\n'
        f'Context:\n{context_str}\n\n'
        f'Answer:'
    )

    resp = client.messages.create(
        model=model_name,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': user_msg}],
    )
    return resp.content[0].text.strip()

print('generate_answer() הוגדרה ✓')

---
## שלב 8 — הפונקציה הראשית: `answer()`

> 📋 **Required System Interface — דרישה ישירה מהמסמך:**
> ```python
> def answer(question: str) -> dict:
>     return {
>         'answer':           str,
>         'sources':          list[str],
>         'retrieved_chunks': list[dict]
>     }
> ```
> "This interface follows the idea of a fixed `eval_runner.py` that can evaluate all student systems through the same `answer()` function."

### `answer(question, k=5, strategy='fixed')`

**קלט:**
- `question` — שאלת המשתמש
- `k` — מספר קטעים לחיפוש (ברירת מחדל: 5)
- `strategy` — אסטרטגיית chunking לחיפוש

**תהליך:**
```
Step 1: retrieve(question, k, strategy)
        → top-k chunks with scores

Step 2: generate_answer(question, retrieved)
        → text with [Source: ...] citation

Step 3: parse sources with regex
        r'\[Source:\s*(.*?)\]'  → list[str]
        fallback: all retrieved chunk_ids
```

**פלט מלא — בדיוק כנדרש:**
```json
{
  "answer": "The discharge criteria include...[Source: aap_p0251_fixed_000]",
  "sources": ["aap_p0251_fixed_000"],
  "retrieved_chunks": [
    {
      "chunk_id": "aap_p0251_fixed_000",
      "text":     "...",
      "score":    0.874,
      "metadata": {"source": "AAP...pdf", "page": 251}
    }
  ]
}
```

In [ ]:
# ─────────────────────────────────────────────────────────────
# Required Interface (דרישה מפורשת מהמסמך)
# ─────────────────────────────────────────────────────────────
def answer(question: str, k: int = 5, strategy: str = 'fixed') -> dict:
    """
    הפונקציה הראשית של ה-RAG pipeline.

    ממלאת את ה-Required System Interface שנדרש במסמך המטלה:
        answer(question: str) -> {
            'answer':           str,
            'sources':          list[str],
            'retrieved_chunks': list[dict]
        }

    קלט:
        question — שאלה בשפה טבעית
        k        — מספר קטעים לאחזר (ברירת מחדל: 5)
        strategy — 'fixed' | 'paragraph'

    תהליך:
        1. retrieve()        → top-k chunks              [דרישה 7]
        2. generate_answer() → grounded answer + [Source] [דרישות 8+9]
        3. parse sources     → list[str] of chunk IDs    [דרישה 9]
    """
    # שלב 1: Retrieval (דרישה 7)
    retrieved = retrieve(question, k=k, strategy=strategy)

    # שלב 2: Generation + citation (דרישות 8+9)
    answer_text = generate_answer(question, retrieved)

    # שלב 3: חילוץ מקורות מ-[Source: chunk_id_1, chunk_id_2]
    match = re.search(r'\[Source:\s*(.*?)\]', answer_text, re.IGNORECASE)
    if match:
        sources = [s.strip() for s in re.split(r'[,;]', match.group(1)) if s.strip()]
    else:
        sources = [c['chunk_id'] for c in retrieved]   # fallback: כל מה שנמצא

    return {
        'answer':           answer_text,
        'sources':          sources,
        'retrieved_chunks': retrieved,
    }

print('answer() — Required Interface — הוגדרה ✓')

In [ ]:
# ─── הדגמה: 5 שאלות מסוגים שונים ───
demo_questions = [
    ('factual',     'What are the discharge criteria for a child with febrile seizure?'),
    ('numerical',   'What is the minimum pulse oximetry monitoring duration for higher-risk BRUE infants?'),
    ('negation',    'Is a sunken fontanel associated with hydrocephalus?'),
    ('comparison',  'How does acute rhinorrhea differ from chronic rhinorrhea in duration?'),
    ('not-in-corpus', 'What is the capital of France?'),  # ← צפוי: "not found"
]

for cat, q in demo_questions:
    print(f'\n[{cat}] {q}')
    res = answer(q, k=5)
    print(f'→ {res["answer"][:200]}')
    print(f'  Sources: {res["sources"][:2]}')

---
## שלב 10 — הערכה (Evaluation)

> 📋 **דרישות 7+10 — Evaluation:**  
> **Retrieval evaluation:** "Measure whether the system retrieves the correct chunks."  
> **Answer evaluation:** "Manually inspect at least 10 answers and classify them as:
> Correct / Partially correct / Incorrect / Unsupported/hallucinated"
> 
> **Gold set:** `eval/gold_set.jsonl` — 50 שאלות ב-5 קטגוריות.

### `soft_hit(retrieved, gold_ids)` — מדד Hit@k

**קלט:**
- `retrieved` — list of chunks שהמערכת מצאה
- `gold_ids` — list of `must_cite_chunk_ids` מה-gold set

**לוגיקה:**
```
1. exact match: retrieved_ids ∩ gold_ids ≠ ∅  → Hit!
2. soft match:  same source prefix (aap/kliegman)
               + page number within ±2  → Hit!

למה soft match?
   Gold chunk IDs נבנו ידנית לפני הרצת הindex.
   גבולות chunk יכולים לזוז ±1 עמוד — soft match מונע False Negative.
```

### `run_evaluation()` — מחשב Hit@k ו-Precision@k

| מדד | הגדרה | ציפיה |
|---|---|---|
| **Hit@k** | fraction of questions with ≥1 correct chunk in top-k | > 0.70 |
| **Precision@k** | fraction of top-k chunks from correct source | > 0.50 |
| **Mean latency** | average seconds per question | < 5s |

In [ ]:
# ─────────────────────────────────────────────────────────────
# דרישה 10: Evaluation
# ─────────────────────────────────────────────────────────────
def extract_pages(chunk_ids):
    """מחלץ set של מספרי עמוד מ-chunk_id כמו 'aap_p0051_fixed_000' → {51}"""
    return {int(p[1:]) for cid in chunk_ids
            for p in cid.split('_') if p.startswith('p') and p[1:].isdigit()}

def extract_sources(chunk_ids):
    """מחלץ prefixes: {'aap', 'kliegman'}"""
    return {cid.split('_')[0] for cid in chunk_ids}


def soft_hit(retrieved: list, gold_ids: list) -> bool:
    """
    מחזיר True אם המערכת מצאה מקור נכון.

    שיטה 1 — exact match: chunk_id מופיע ב-gold_ids
    שיטה 2 — soft match:  אותו prefix (aap/kliegman)
                           + מספר עמוד בהפרש ≤ 2

    Soft match מכסה:
    - גבולות chunk שזעו עמוד אחד
    - gold IDs שנכתבו ידנית לפני בניית ה-index
    """
    if {c['chunk_id'] for c in retrieved} & set(gold_ids):
        return True
    gold_pages   = extract_pages(gold_ids)
    gold_sources = extract_sources(gold_ids)
    for c in retrieved:
        src  = c['metadata'].get('doc_id_prefix', '')
        page = int(c['metadata'].get('page', 0))
        if src in gold_sources and any(abs(page - gp) <= 2 for gp in gold_pages):
            return True
    return False


def run_evaluation(gold: list, k: int = 5, strategy: str = 'fixed',
                   limit: int = None, verbose: bool = True) -> dict:
    """
    מריץ gold set דרך ה-pipeline ומחשב מדדים.

    קלט:
        gold     — list of gold questions (מ-gold_set.jsonl)
        k        — top-k לחיפוש
        strategy — 'fixed' | 'paragraph'
        limit    — מגבלת שאלות לבדיקה
        verbose  — הדפס 3 דוגמאות ראשונות

    פלט: dict עם hit_rate, precision, mean_latency

    מדדים:
        Hit@k       — fraction of questions where soft_hit() = True
        Precision@k — fraction of top-k from correct source
        Mean latency — ממוצע שניות לשאלה
    """
    if limit: gold = gold[:limit]
    hits, precs, lats, cat_hits = [], [], [], {}

    for i, item in enumerate(gold):
        t0  = time.time()
        res = answer(item['question'], k=k, strategy=strategy)
        lats.append(time.time() - t0)

        hit = soft_hit(res['retrieved_chunks'], item.get('must_cite_chunk_ids', []))
        hits.append(int(hit))

        gold_src = extract_sources(item.get('must_cite_chunk_ids', []))
        precs.append(sum(1 for c in res['retrieved_chunks']
                         if c['metadata'].get('doc_id_prefix','') in gold_src) / len(res['retrieved_chunks']))

        cat = item.get('category', 'unknown')
        cat_hits.setdefault(cat, []).append(int(hit))

        if verbose and i < 3:
            print(f'[{i+1}] {"✓" if hit else "✗"}  {item["question"][:65]}')
            print(f'     {res["answer"][:120]}...')

    print(f'\nHit@{k}:       {sum(hits)/len(hits):.3f}  ({sum(hits)}/{len(hits)})')
    print(f'Precision@{k}: {statistics.mean(precs):.3f}')
    print(f'Latency:      {statistics.mean(lats):.2f}s')
    print('\nPer category:')
    for cat, ch in sorted(cat_hits.items()):
        print(f'  {cat:<15}  {sum(ch)/len(ch):.2f}  ({sum(ch)}/{len(ch)})')
    return {'hit_rate': sum(hits)/len(hits), 'precision': statistics.mean(precs)}

print('פונקציות הערכה הוגדרו ✓')

In [ ]:
# טעינת gold set
gold_path = PROJECT_ROOT / 'eval' / 'gold_set.jsonl'
with open(gold_path, encoding='utf-8') as f:
    gold_set = [json.loads(l) for l in f if l.strip()]

print(f'Gold set: {len(gold_set)} שאלות')
cats = Counter(q['category'] for q in gold_set)
print('קטגוריות:', dict(cats))

# הערכה על 20 שאלות (לחסוך API calls)
print('\n=== הרצת evaluation על 20 שאלות ===')
metrics = run_evaluation(gold_set, k=5, strategy='fixed', limit=20)

---
## שלב 11 — Ablation Study

> 📋 **דרישה — Ablation Study:**  
> "Run at least **two small ablation experiments**."  
> "Report the result in a table: Experiment | Hit@5 | Answer accuracy | Notes"
> 
> דוגמאות מהמסמך:
> - chunk size 300 vs 700
> - top-k = 3 vs top-k = 8
> - embedding model A vs B
> - dense vs BM25 vs hybrid

### ה-4 ניסויים שנריץ:

| ניסוי | שינוי | השערה |
|---|---|---|
| Baseline | fixed 400, k=5 | — |
| k=3 | פחות קטעים | recall יורד, precision עולה |
| k=8 | יותר קטעים | recall עולה, noise עולה |
| chunk 300 | קטעים קצרים | טוב לשאלות עובדתיות קצרות |
| paragraph | פסקאות | טוב לשאלות AAP |

In [ ]:
# בניית index נוסף עם chunk_size=300
print('בונה index עם chunk_size=300...')
small_chunks     = chunk_fixed_size(all_docs, chunk_size=300, overlap=30)
small_embeddings = embed_chunks(model, small_chunks)
col_small = upsert_collection(chroma_client, 'pediatric_small300',
                               small_chunks, small_embeddings)
print('✓ index 300-char מוכן')

In [ ]:
# ─────────────────────────────────────────────────────────────
# Ablation Study — דרישה ישירה מהמסמך
# ─────────────────────────────────────────────────────────────
def ablation_retrieve(query: str, k: int, col) -> list:
    """
    גרסה מינימלית של retrieve() לשימוש באבלציה.
    מקבלת collection object ישירות (לא strategy string)
    כדי לאפשר בדיקה של collections שונים (כולל 300-char).
    """
    q_vec = model.encode([query], normalize_embeddings=True).tolist()
    r = col.query(query_embeddings=q_vec, n_results=k,
                  include=['documents', 'metadatas', 'distances'])
    return [{'chunk_id': cid, 'text': t,
             'score': round(1.0 - d/2.0, 4), 'metadata': m}
            for cid, t, m, d in zip(r['ids'][0], r['documents'][0],
                                     r['metadatas'][0], r['distances'][0])]


EVAL_N = 15
gold_sample = gold_set[:EVAL_N]

experiments = [
    ('fixed 400, k=5 (baseline)', col_fixed, 5),
    ('fixed 400, k=3',            col_fixed, 3),
    ('fixed 400, k=8',            col_fixed, 8),
    ('fixed 300, k=5',            col_small, 5),
    ('paragraph, k=5',            col_para,  5),
]

notes = {
    'fixed 400, k=5 (baseline)': '',
    'fixed 400, k=3':            'פחות recall',
    'fixed 400, k=8':            'יותר noise',
    'fixed 300, k=5':            'טוב לשאלות קצרות',
    'paragraph, k=5':            'טוב ל-AAP Q&A',
}

print(f'\n{"Experiment":<30} {"Hit@k":>8} {"Prec@k":>8}  Notes')
print('-' * 70)

for label, col, k in experiments:
    hits, precs = [], []
    for item in gold_sample:
        retrieved = ablation_retrieve(item['question'], k, col)
        hits.append(int(soft_hit(retrieved, item.get('must_cite_chunk_ids', []))))
        gold_src = extract_sources(item.get('must_cite_chunk_ids', []))
        precs.append(sum(1 for c in retrieved
                         if c['metadata'].get('doc_id_prefix','') in gold_src) / len(retrieved))
    print(f'{label:<30} {sum(hits)/len(hits):>8.3f} {statistics.mean(precs):>8.3f}  {notes[label]}')

---
## שלב 12 — ניתוח כשלים (Failure Analysis)

> 📋 **דרישה 10 — Error Analysis:**  
> "manually inspect at least 10 answers and classify them as:
> Correct / Partially correct / Incorrect / Unsupported/hallucinated"

### טקסונומיית הכשלים ב-RAG:

| סוג | תסמין | גורם | פתרון |
|---|---|---|---|
| **Retrieval miss** | ✗ chunk נכון לא חוזר | vocabulary mismatch, chunk קטן מדי | הגדל k, BM25 hybrid |
| **Wrong granularity** | ✓ עמוד נכון, ✗ לא הקטע הספציפי | chunk גדול מדי | הקטן chunk_size |
| **Hallucination** | score נמוך → LLM ממלא ממידע כללי | system prompt חלש | הדק את כלל ה-"only from context" |
| **Too many chunks** | תשובה מבולבלת | k גדול מביא noise | הקטן k, הוסף reranker |
| **Language gap** | שאלה בניסוח שונה מהקורפוס | embedding לא מספיק סמנטי | query expansion |

In [ ]:
# ניתוח ידני — 10 תשובות ראשונות
# לאחר הרצה: סווג כל תשובה: Correct / Partially / Incorrect / Hallucinated

print('=== ניתוח ידני — 10 תשובות ===\n')

for i, item in enumerate(gold_set[:10]):
    res = answer(item['question'], k=5)
    print(f'[{i+1:02d}] {item["category"]}')
    print(f'  שאלה:    {item["question"][:70]}')
    print(f'  ציפיה:   {item["reference_answer"][:80]}')
    print(f'  תשובה:   {res["answer"][:80]}')
    print(f'  score:   {res["retrieved_chunks"][0]["score"]:.3f}')
    print(f'  מקורות:  {res["sources"][:2]}')
    print(f'  → סיווג: [ Correct | Partially | Incorrect | Hallucinated ]')
    print()

---
## סיכום: מיפוי מלא דרישות → קוד

```
דרישה 1  — Data collection
   ↓  setup_data.py + שני PDFs ב-data/raw/

דרישה 2  — Document Loading
   ↓  load_pdf(path, prefix) → [{doc_id, text, metadata}]

דרישה 3  — Text Cleaning
   ↓  clean_text(text)  — regex-based noise removal

דרישה 4  — Chunking (2 strategies)
   ↓  chunk_fixed_size(docs, 400, 50) → fixed chunks
   ↓  chunk_paragraph(docs, 100, 700) → paragraph chunks
   ↓  שמירה ב-data/processed/*.jsonl

דרישה 5  — Embedding
   ↓  embed_chunks(model, chunks)  — SentenceTransformer all-MiniLM-L6-v2

דרישה 6  — Indexing
   ↓  upsert_collection(client, name, chunks, embeddings)
   ↓  ChromaDB PersistentClient → index/ (reproducible)

דרישה 7  — Retrieval + metric
   ↓  retrieve(query, k, strategy) → [{chunk_id, text, score, metadata}]
   ↓  Hit@k, Precision@k ב-run_evaluation()

דרישה 8  — Answer Generation
   ↓  generate_answer(question, chunks) → Claude claude-haiku-4-5
   ↓  SYSTEM_PROMPT: grounded + no hallucination

דרישה 9  — Citations
   ↓  [Source: chunk_id] parse ב-answer()
   ↓  sources: list[str] בפלט

דרישה 10 — Evaluation + Error Analysis
   ↓  run_evaluation(gold_set, k, strategy)
   ↓  ablation: 5 ניסויים → טבלה
   ↓  manual: 10 תשובות → סיווג ידני
```

---
### מה אפשר לשפר?

1. **Cross-encoder reranker** — `cross-encoder/ms-marco-MiniLM-L-6-v2` — precision++
2. **Hybrid BM25 + dense** — TF-IDF לsparks שdeep embedding מפספס
3. **Query expansion** — 3 גרסאות לשאלה, union של תוצאות
4. **HyDE** — LLM כותב "תשובה בדיונית", embed אותה, חפש על פיה
5. **Better embedder** — `BAAI/bge-base-en-v1.5` (110M params, MTEB 56.0)